# DS 3010 ? Sleep Health & Lifestyle Project (Starter Notebook)
Team: **Nina De Grandis** & **Lucas Sorge**

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    r2_score, mean_squared_error, mean_absolute_error
)


## 2. Data Access

In [ ]:
# Check if kagglehub is installed and available
try:
    import kagglehub
    HAS_KAGGLEHUB = True
except ImportError:
    HAS_KAGGLEHUB = False

import os

DATA_DIR = None
CSV_PATH = None

if HAS_KAGGLEHUB:
    path = kagglehub.dataset_download("uom190346a/sleep-health-and-lifestyle-dataset")
    print("Dataset dir:", path)
    print("Files:", os.listdir(path))
    # Try to locate a CSV automatically
    csv_candidates = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
    if csv_candidates:
        CSV_PATH = os.path.join(path, csv_candidates[0])
        print("Using CSV:", CSV_PATH)
    else:
        print("No CSV file found in the KaggleHub directory.")
else:
    print("kagglehub not available; use option 2B and set CSV_PATH manually.")

## 3. Load Data

In [ ]:
assert CSV_PATH is not None, "CSV_PATH is not set. Complete Section 2A or 2B first."
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## 4. Quick EDA

In [ ]:
df.info()

df.describe(include="all").T

# Missing values overview
df.isnull().sum().sort_values(ascending=False)

# Simple histogram for numeric columns (one-by-one to avoid clutter)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    plt.figure()
    df[col].hist(bins=30)
    plt.title(f"Distribution: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

## 5. Define Targets & Feature Matrix

In [ ]:
# Classification target: Sleep Disorder
y_cls = df["Sleep Disorder"]
X_cls = df.drop(columns=["Sleep Disorder"])

# Regression target: Quality of Sleep
y_reg = df["Quality of Sleep"]
X_reg = df.drop(columns=["Quality of Sleep"])

# Identify numeric & categorical columns from the classification set
num_cols = X_cls.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_cls.columns if c not in num_cols]

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

## 6. Preprocessing Pipelines

In [ ]:
# Numeric: impute median + scale
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: impute most_frequent + one-hot
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ]
)

## 7. Train/Test Split

In [ ]:
# Make clean copies of X/y for both tasks, removing NaNs only from targets
# Classification
mask_cls = df["Sleep Disorder"].notna()
X_cls = df.loc[mask_cls].drop(columns=["Sleep Disorder"])
y_cls = df.loc[mask_cls, "Sleep Disorder"]

# Regression
mask_reg = df["Quality of Sleep"].notna()
X_reg = df.loc[mask_reg].drop(columns=["Quality of Sleep"])
y_reg = df.loc[mask_reg, "Quality of Sleep"]

print("Target NaNs removed ->",
      "cls:", (~mask_cls).sum(), "rows dropped |",
      "reg:", (~mask_reg).sum(), "rows dropped")

# sanity check class balance
print("Class distribution:\n", y_cls.value_counts(dropna=False))

# If each class has at least 2 samples, we can stratify safely
from sklearn.model_selection import train_test_split

ok_to_stratify = (y_cls.value_counts().min() >= 2)
strat = y_cls if ok_to_stratify else None

Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=strat
)

Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Classification shapes:", Xtr_c.shape, Xte_c.shape, ytr_c.shape, yte_c.shape)
print("Regression shapes:", Xtr_r.shape, Xte_r.shape, ytr_r.shape, yte_r.shape)

## 8. Baseline Models

### 8.1 Logistic Regression ? Sleep Disorder (Classification)

In [ ]:
log_pipe = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])
log_pipe.fit(Xtr_c, ytr_c)
yp_c = log_pipe.predict(Xte_c)

print("Accuracy:", accuracy_score(yte_c, yp_c))
print("\nClassification Report\n", classification_report(yte_c, yp_c))
print("\nConfusion Matrix\n", confusion_matrix(yte_c, yp_c))

### 8.2 Linear Regression ? Quality of Sleep (Regression)

In [ ]:
# Safety: drop obvious identifiers if still present in the split frames
for id_like in ["Person ID", "PersonID", "ID"]:
    if id_like in Xtr_r.columns:
        Xtr_r = Xtr_r.drop(columns=[id_like])
        Xte_r = Xte_r.drop(columns=[id_like])

# Build a regression preprocessor that AUTO-selects columns from whatever X you pass
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_reg = ColumnTransformer([
    ("num", num_pipe, selector(dtype_include=np.number)),
    ("cat", cat_pipe, selector(dtype_exclude=np.number)),
])

lin_pipe = Pipeline([
    ("pre", preprocess_reg),
    ("reg", LinearRegression())
])

lin_pipe.fit(Xtr_r, ytr_r)
yp_r = lin_pipe.predict(Xte_r)

print("R^2:", r2_score(yte_r, yp_r))
rmse = np.sqrt(mean_squared_error(yte_r, yp_r))
print("RMSE:", rmse)
print("MAE:", mean_absolute_error(yte_r, yp_r))

# quick residuals sanity check for the report
import matplotlib.pyplot as plt
resid = yte_r - yp_r
plt.figure()
plt.scatter(yp_r, resid, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Quality of Sleep")
plt.ylabel("Residual (y_true - y_pred)")
plt.title("Residuals vs Predicted (Linear Regression)")
plt.show()

## 9. Model Insight: Coefficients

In [ ]:
# Recover feature names after preprocessing (for interpretability)
ohe = log_pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)
feature_names = np.concatenate([num_cols, cat_feature_names])

if hasattr(log_pipe.named_steps["clf"], "coef_"):
    coefs = log_pipe.named_steps["clf"].coef_
    # For multinomial, this is (n_classes, n_features). We'll display per class.
    if coefs.ndim == 2 and coefs.shape[0] > 1:
        for i, row in enumerate(coefs):
            top_idx = np.argsort(np.abs(row))[::-1][:15]
            print(f"Top 15 coefficients for class {i}:")
            for idx in top_idx:
                print(f"{feature_names[idx]:40s} {row[idx]: .4f}")
            print("-"*60)
    else:
        row = coefs.ravel()
        top_idx = np.argsort(np.abs(row))[::-1][:20]
        print("Top 20 coefficients (binary or OvR view):")
        for idx in top_idx:
            print(f"{feature_names[idx]:40s} {row[idx]: .4f}")
else:
    print("Classifier does not expose coefficients.")

## 10. Save Artifacts (Pipelines & Metrics)

In [ ]:
from pathlib import Path
import json
import joblib

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

# Save pipelines
joblib.dump(log_pipe, out_dir / "logistic_sleep_disorder_pipeline.joblib")
joblib.dump(lin_pipe, out_dir / "linear_sleep_quality_pipeline.joblib")

# Save quick metrics
metrics = {
    "classification": {
        "accuracy": float(accuracy_score(yte_c, yp_c))
    },
    "regression": {
        "r2": float(r2_score(yte_r, yp_r)),
        "rmse": float(rmse),
        "mae": float(mean_absolute_error(yte_r, yp_r))
    }
}
with open(out_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", list(out_dir.iterdir()))